In [0]:
df = spark.table("bankingdata.bronze.customers")

In [0]:
df.columns

### Handling rescue data column

In [0]:
from pyspark.sql.functions import *

In [0]:
df_rescued = df.filter(col("_rescued_data").isNotNull())
df = df.filter(col("_rescued_data").isNull())

In [0]:
df_rescued_final = df_rescued.select(
    col("CustomerID"),          
    col("_rescued_data"),       
    col("source_file"),        
    col("ingestion_time")       
)

In [0]:
df_rescued_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bankingdata.rescueddata.customers_rescued")

### Required Columns

In [0]:
df = df.select(
    col("CustomerID"),
    col("FirstName"),
    col("LastName"),
    col("City"),
    col("Email"),
    col("Phone"),
    col("ModifiedDate")
)

### Schema

In [0]:
df = df.select(
    col("CustomerID").cast("int").alias("CustomerID"),
    col("FirstName").cast("string").alias("FirstName"),
    col("LastName").cast("string").alias("LastName"),
    col("City").cast("string").alias("City"),
    col("Email").cast("string").alias("Email"),
    col("Phone").cast("string").alias("Phone"),
    col("ModifiedDate").cast("timestamp").alias("ModifiedDate"),
)

### Primary Key Validation

In [0]:
df = df.filter(
    col("CustomerID").isNotNull() &         
    (trim(col("CustomerID").cast("string")) != "") & 
    (col("CustomerID") != 0)                
)

### Business Quality Check

In [0]:
df = df.filter(
    # CustomerID
    col("CustomerID").isNotNull() &
    (col("CustomerID") > 0) &

    # FirstName
    col("FirstName").isNotNull() &
    (length(trim(col("FirstName"))) > 1) &
    col("FirstName").rlike("^[A-Za-z]+$") &

    # LastName
    col("LastName").isNotNull() &
    (length(trim(col("LastName"))) > 1) &
    col("LastName").rlike("^[A-Za-z]+$") &

    # City
    col("City").isNotNull() &
    (length(trim(col("City"))) > 2) &
    ~col("City").rlike("^[0-9]+$") &

    # Email
    col("Email").isNotNull() &
    col("Email").rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$") &

    # Phone
    col("Phone").isNotNull() &
    col("Phone").rlike("^[0-9]{10}$") &

    # ModifiedDate
    col("ModifiedDate").isNotNull() &
    (col("ModifiedDate") <= current_timestamp())
)

### Remove Duplicates

In [0]:
from pyspark.sql.window import Window

# Define window
window_spec = Window.partitionBy("CustomerID") \
                    .orderBy(col("ModifiedDate").desc())

# Apply deduplication
df_dedup = df.withColumn("rn", row_number().over(window_spec)) \
             .filter(col("rn") == 1) \
             .drop("rn")

### Delta table

In [0]:
# save table
silver_table = "bankingdata.silver.customer"

df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(silver_table)

print("Silver table created:", silver_table)

In [0]:
%sql
select * from bankingdata.silver.customer limit 3